# Stage 3 — Pooled HMM Regime Features

Fit one global `GaussianHMM` on all training-auction `micro_factor` sequences.  
Each auction day is treated as an independent chain (per-day lengths passed to HMM).

**Causal features extracted:**
- `regime_eod` — Viterbi-decoded state at end of day
- `prob_regime_k_eod` — **filtered** (forward-algorithm) posteriors at EOD  
  *(never `model.predict_proba()` which uses future data)*
- `minutes_in_post_regime` — persistence of EOD state in post-auction window

> **Leakage note:** `fit_pooled_hmm` is called on train-only rows inside every CV fold in Stage 6.
> The full-sample fit here is for exploratory inspection only.

**Reads:** `data/cache/intraday_stage1.parquet`  
**Writes:** `data/cache/regime_stage3.parquet`

Ref: Hamilton (1989).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import CACHE_DIR, N_REGIMES
from src.regime import HAVE_HMM, fit_pooled_hmm, regime_features_per_auction

if not HAVE_HMM:
    print('hmmlearn not installed — regime stage will be skipped in CV.')

intraday_df = pd.read_parquet(CACHE_DIR / 'intraday_stage1.parquet')
print(f'Loaded: {len(intraday_df):,} rows  |  {intraday_df["auction_id"].nunique()} auctions')

## 3a. Fit HMM (full-sample, exploratory)

In [ ]:
if HAVE_HMM:
    hmm_model = fit_pooled_hmm(intraday_df)
    print(f'\nTransition matrix ({N_REGIMES}×{N_REGIMES}):')
    print(hmm_model.transmat_.round(3))
    print('\nState means (micro_factor):', hmm_model.means_.flatten().round(4))
    print('State variances:', hmm_model.covars_.flatten().round(6))
else:
    print('Skipped — hmmlearn unavailable.')
    hmm_model = None

## 3b. Extract per-auction regime features

In [ ]:
if hmm_model is not None:
    regime_df = regime_features_per_auction(intraday_df, hmm_model)
    print(regime_df.shape)
    regime_df.head()
else:
    regime_df = pd.DataFrame({'auction_id': intraday_df['auction_id'].unique()})
    print('Empty regime_df created.')

## 3c. Diagnostics

In [ ]:
if hmm_model is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Regime state distribution
    regime_df['regime_eod'].value_counts().sort_index().plot.bar(ax=axes[0], rot=0)
    axes[0].set_title(f'EOD regime state distribution (K={N_REGIMES})')
    axes[0].set_xlabel('Regime')

    # Filtered prob at EOD — one example auction
    prob_cols = [f'prob_regime_{k}_eod' for k in range(N_REGIMES)]
    regime_df[prob_cols].plot(ax=axes[1], marker='o', markersize=3, linewidth=0.8)
    axes[1].set_title('Filtered regime prob at EOD (all auctions)')
    axes[1].set_xlabel('Auction index')

    plt.tight_layout()
    plt.show()

    print('Regime transitions:')
    print(regime_df['regime_transition'].value_counts().head(10))

## 3d. Persist

In [ ]:
regime_df.to_parquet(CACHE_DIR / 'regime_stage3.parquet', index=False)
print('Saved → cache/regime_stage3.parquet')